# Decision Trees

Basics (Daniel?):

- What is a tree?
- Root, Decision Node and Leaf
- Small Example
- Pros and Cons compared to Linear Regression etc.

Concepts (Clemens?):

- Different Algorithms (CART, ID3 etc.)
- Entropy
- Gini impurity
- Information Gain
- Pruning


In [191]:
import pandas as pd

In [192]:
df = pd.DataFrame(
    [
        ["Green", 3, "Apple"],
        ["Yellow", 3, "Apple"],
        ["Red", 1, "Grape"],
        ["Red", 1, "Grape"],
        ["Yellow", 3, "Lemon"],
    ],
    columns=["color", "diameter", "fruit"],
)

In [193]:
class Question:
    def __init__(self, df, column_index, value):
        self.df = df
        self.column_index = column_index
        self.column_name = df.columns[column_index]
        self.value = value

    def match(self, example):
        val = example[self.column_name]

        if isinstance(self.value, (int, float)):  # Numeric
            return val >= self.value
        return val == self.value  # Other

    def __repr__(self):
        condition = ">=" if isinstance(self.value, (int, float)) else "=="
        return f"{self.column_name} {condition} {self.value}?"

In [194]:
q = Question(df, 0, "Yellow")

print('["Green", 3, "Apple"]: ', q.match(df.iloc[0]))
print('["Yellow", 3, "Apple"]: ', q.match(df.iloc[1]))

["Green", 3, "Apple"]:  False
["Yellow", 3, "Apple"]:  True


In [195]:
def partition(df, question):
    mask = df.apply(lambda row: question.match(row), axis=1)

    matches = df[mask]
    not_matches = df[~mask]

    return matches, not_matches


# Usage:
true_rows, false_rows = partition(df, q)
print(q)
true_rows

color == Yellow?


,color,diameter,fruit
1,Yellow,3,Apple
4,Yellow,3,Lemon


**Calculate Gini Impurity [(source)](https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity):**
J... Labels
p... Probability of randomly selecting label
$$Gini = 1 - \sum_{i=1}^{J} * p^2_i$$


In [196]:
def gini(df_col):
    # automatically calculates probablities for each label
    probs = df_col.value_counts(normalize=True)
    return 1 - (probs**2).sum()


# 0% Chance of misclassification
print('Gini for ["Apple", "Apple"]:', gini(pd.DataFrame(["Apple", "Apple"])))

# 50% Chance of misclassification
print('Gini for ["Apple", "Grape"]:', gini(pd.DataFrame(["Apple", "Grape"])))

# 66.7% Chance of misclassification a random example from the dataset
print(
    'Gini for ["Apple", "Grape", "Peach"]:',
    gini(pd.DataFrame(["Apple", "Grape", "Peach"])),
)

Gini for ["Apple", "Apple"]: 0.0
Gini for ["Apple", "Grape"]: 0.5
Gini for ["Apple", "Grape", "Peach"]: 0.6666666666666667


In [197]:
def information_gain(left, right, current_uncertainty):
    n_left = len(left)
    n_right = len(right)
    total = n_left + n_right

    if total == 0:
        return 0

    # proportion for left side
    p = n_left / total

    gain = current_uncertainty - (p * gini(left)) - ((1 - p) * gini(right))

    return gain

In [198]:
# We can calculate information gain as follows
current_uncertainty = gini(df["fruit"])
true_rows, false_rows = partition(df, Question(df, 0, "Green"))
ig_green = information_gain(
    left=true_rows["fruit"],
    right=false_rows["fruit"],
    current_uncertainty=current_uncertainty,
)

true_rows, false_rows = partition(df, Question(df, 0, "Red"))
ig_red = information_gain(
    left=true_rows["fruit"],
    right=false_rows["fruit"],
    current_uncertainty=current_uncertainty,
)

print("Information Gain for Green:", ig_green)
print("Information Gain for Red:", ig_red)

Information Gain for Green: 0.1399999999999999
Information Gain for Red: 0.37333333333333324


In [199]:
def find_best_split(df, label_column):
    best_gain = 0
    best_question = None
    current_uncertainty = gini(df[label_column])  # ... with respect to the label
    n_features = len(df.columns) - 1  # cols - label col

    for col_idx in range(n_features):
        # Try split for all values in column (e.g. Red, Yellow...)
        values = df.iloc[:, col_idx].unique()

        for val in values:
            question = Question(df, col_idx, val)
            true_rows, false_rows = partition(df, question)

            if len(true_rows) == 0 or len(false_rows) == 0:
                continue

            # information gain from this split, with respect to the label column (e.g fruit)
            gain = information_gain(
                true_rows[label_column], false_rows[label_column], current_uncertainty
            )

            # Keep Track of best gain so far
            if gain >= best_gain:
                best_gain, best_question = gain, question

    return best_gain, best_question

In [200]:
find_best_split(df, "fruit")

(np.float64(0.37333333333333324), diameter == 1?)

In [201]:
class Leaf:
    def __init__(self, df_rows, label_column):
        self.predictions = df_rows[label_column].value_counts().to_dict()


class Decision_Node:
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

In [202]:
def build_tree(df, label_column):
    gain, question = find_best_split(df, label_column)

    # No more decision to be performed, dead-end
    if gain == 0:
        return Leaf(df, label_column)

    # Ask best question
    true_rows, false_rows = partition(df, question)

    # Recursively building Tree
    true_branch = build_tree(true_rows, label_column)
    false_branch = build_tree(false_rows, label_column)

    return Decision_Node(question, true_branch=true_branch, false_branch=false_branch)

In [203]:
def print_tree(node, spacing=""):
    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        total = sum(node.predictions.values())
        # Calculate percentages for more intuitive reading
        summary = {
            k: f"{round((v / total) * 100, 1)}%" for k, v in node.predictions.items()
        }
        print(spacing + "└── Prediction: " + str(summary))
        return

    # Print the question at the current node
    print(spacing + "Query: " + str(node.question))

    # Handle the True branch
    print(spacing + "├── True:")
    print_tree(node.true_branch, spacing + "│   ")

    # Handle the False branch
    print(spacing + "└── False:")
    print_tree(node.false_branch, spacing + "    ")

In [204]:
# Not really interesting yet
tree = build_tree(df, "fruit")
print_tree(tree)

Query: diameter == 1?
├── True:
│   └── Prediction: {'Grape': '100.0%'}
└── False:
    Query: color == Yellow?
    ├── True:
    │   └── Prediction: {'Apple': '50.0%', 'Lemon': '50.0%'}
    └── False:
        └── Prediction: {'Apple': '100.0%'}


In [205]:
def classify(row, node):
    # base-case, leaf can return its predictions
    if isinstance(node, Leaf):
        return node.predictions

    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

In [206]:
test_df = pd.DataFrame(
    [
        ["Red", 1],  # Easy example: Grape
        ["Yellow", 3],  # Could either be Apple or Lemon
        ["Green", 5],  # How does it handle new Values - probably Apple?
        ["Blabla", -2],  # Faulty data
    ],
    columns=["color", "diameter"],
)

# Grape
print(classify(test_df.iloc[0], tree))

# Returns both Apple and lemon => we need to decide ourselves (e.g. random)
print(classify(test_df.iloc[1], tree))

# Apple
print(classify(test_df.iloc[2], tree))

# Whatever this is
print(classify(test_df.iloc[3], tree))

{'Grape': 2}
{'Apple': 1, 'Lemon': 1}
{'Apple': 1}
{'Apple': 1}


Sources:

- (Video)[https://www.youtube.com/watch?v=LDRbO9a6XPU]
- (Wikipedia)[https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity]
- (Decision tree - geeksforgeeks)[https://www.geeksforgeeks.org/machine-learning/decision-tree-implementation-python/]
- (Sklearn)[https://scikit-learn.org/stable/modules/tree.html]
